# MCP project setup

A document chatbot has two connections: the application talks to Claude through
its model provider, and an MCP client talks to a local document server over stdio.
The server owns a small in-memory document dictionary. These are sample strings,
not actual PDF or Word files.

Start with the lesson's baseline: a chat with **no connected MCP clients**, so an
answer cannot have come from a document tool. Then smoke-test the completed
backend separately. Tools, resources and prompts are already implemented in the
shared application modules; the later lessons will explain those parts.

Jupyter already runs an event loop. Use top-level `await`; do not call the
interactive CLI or `asyncio.run()` from a notebook cell.

## Setup

Load the course root's `.env`, use its current provider, and import the shared
application. Launch the MCP server with this kernel's Python executable and an
absolute script path, so it uses the same installed dependencies.

In [1]:
from pathlib import Path
import os
import sys

from dotenv import load_dotenv

module_dir = Path.cwd()
if not (module_dir / "mcp_server.py").is_file():
    module_dir = module_dir / "07_model_context_protocol"
module_dir = module_dir.resolve()
assert (module_dir / "mcp_server.py").is_file()
sys.path.insert(0, str(module_dir))
load_dotenv(module_dir.parent / ".env")

from core.chat import Chat
from core.claude import Claude
from core.sampling import create_sampling_callback
from mcp_client import MCPClient

model = os.getenv("CLAUDE_MODEL", "claude-haiku-4-5")
claude = Claude(model=model)
print("Model:", model)


def tool_calls(chat):
    return [
        block
        for message in chat.messages if message["role"] == "assistant"
        for block in (message["content"] if isinstance(message["content"], list) else [])
        if getattr(block, "type", None) == "tool_use"
    ]

Model: claude-haiku-4-5


## Baseline: a verified answer without document tools

An empty client mapping means no MCP tools are sent to Claude. Check the known
answer before adding another layer.

In [2]:
baseline = Chat(claude_service=claude, clients={})
answer = await baseline.run("What is 1 + 1? Reply with the digit only.")
print(answer)
assert answer.strip() == "2"
assert not tool_calls(baseline)
print("Verified: answer is 2, with no MCP tool calls.")

2
Verified: answer is 2, with no MCP tool calls.


## Ask about documents before connecting a server

The model has not received any document contents. It should ask for them rather
than inventing an account of their contents. Regardless of its wording, the
empty tool-call history proves this reply did not come from an MCP tool.

In [3]:
answer = await baseline.run(
    "What do the documents say? If no documents were supplied, say so "
    "rather than inventing their contents."
)
print(answer)
assert not tool_calls(baseline)
print("Document-tool calls in baseline:", len(tool_calls(baseline)))

No documents were supplied.
Document-tool calls in baseline: 0


## Completed backend smoke check

This goes beyond the initial setup baseline: connect the local server, discover
its capabilities, read a resource, call a read tool, and check an error. Test an
edit only in memory and restore the original value immediately. Retrieve a
formatting prompt without asking the model to execute it.

Finally ask Claude to call `read_doc_contents` and verify the actual `tool_use`
block. The model API runs through the configured provider; the MCP subprocess
runs locally. No filesystem roots are granted.

Keep opening and closing the client inside one async function in one cell:
MCP's task groups must be entered and exited within the same async task.

In [4]:
async def verify_mcp():
    async with MCPClient(
        command=sys.executable,
        args=[str(module_dir / "mcp_server.py")],
        sampling_callback=create_sampling_callback(claude),
    ) as documents:
        tools = await documents.list_tools()
        names = {tool.name for tool in tools}
        assert {"read_doc_contents", "edit_document"} <= names
        print("Registered tools:", sorted(names))

        doc_ids = await documents.read_resource("docs://documents")
        assert "report.pdf" in doc_ids and len(doc_ids) == 6
        original = await documents.read_resource("docs://documents/report.pdf")
        assert original == "The report details the state of a 20m condenser tower."
        print("Document IDs:", doc_ids)
        print("Resource:", original)

        read = await documents.call_tool("read_doc_contents", {"doc_id": "report.pdf"})
        assert not read.isError
        assert any(getattr(item, "text", None) == original for item in read.content)
        missing = await documents.call_tool("read_doc_contents", {"doc_id": "missing.txt"})
        assert missing.isError
        print("Read tool verified; missing document returned an MCP tool error.")

        try:
            edited = await documents.call_tool("edit_document", {
                "doc_id": "report.pdf", "old_str": "20m", "new_str": "25m",
            })
            assert not edited.isError
            assert "25m" in await documents.read_resource("docs://documents/report.pdf")
        finally:
            await documents.call_tool("edit_document", {
                "doc_id": "report.pdf", "old_str": "25m", "new_str": "20m",
            })
        assert await documents.read_resource("docs://documents/report.pdf") == original
        print("In-memory edit verified and restored.")

        prompts = await documents.list_prompts()
        assert "format" in {prompt.name for prompt in prompts}
        prompt_messages = await documents.get_prompt("format", {"doc_id": "report.pdf"})
        assert prompt_messages and "report.pdf" in prompt_messages[0].content.text
        print("Format prompt retrieved; it was not executed.")

        grounded = Chat(claude_service=claude, clients={"documents": documents})
        answer = await grounded.run(
            "Call read_doc_contents with doc_id='report.pdf'. "
            "Then tell me the condenser tower's height from the returned document. "
            "Reply briefly. Do not use the summarize tool."
        )
        calls = tool_calls(grounded)
        assert any(call.name == "read_doc_contents" and call.input.get("doc_id") == "report.pdf"
                   for call in calls)
        assert "20" in answer
        print("Grounded answer:", answer)
        print("Actual model tool calls:", [call.name for call in calls])
    assert documents._session is None
    print("PASS: local MCP server connected, tools/resources/prompts verified, model used a document tool, session closed.")


await verify_mcp()

Registered tools: ['edit_document', 'list_roots', 'read_dir', 'read_doc_contents', 'read_file', 'summarize']
Document IDs: ['deposition.md', 'report.pdf', 'financials.docx', 'outlook.pdf', 'plan.md', 'spec.txt']
Resource: The report details the state of a 20m condenser tower.
Read tool verified; missing document returned an MCP tool error.
In-memory edit verified and restored.
Format prompt retrieved; it was not executed.


Grounded answer: The condenser tower's height is **20 meters**.
Actual model tool calls: ['read_doc_contents']
PASS: local MCP server connected, tools/resources/prompts verified, model used a document tool, session closed.


The first two answers came directly from the model. The final answer used a
real document-tool round trip. The test covers the base document workflow;
sampling, notifications and filesystem roots are included in the shared
application but are not exercised here.

To use the terminal interface outside Jupyter, run from the course root:
`uv run 07_model_context_protocol/main.py`.

Next lesson: defining tools with MCP.